# Cómo está armado este banco

Un Arduino UNO, un sensor magnético de ángulo, un actuador y un motor. Este
notebook recorre el hardware: qué está conectado a qué, por qué está conectado
así, y qué se puede medir con cada parte --incluso mientras alguna de esas partes
todavía no está sobre la mesa.

El banco trabaja en **lazo abierto**: la placa pone un comando de PWM sobre el
actuador y devuelve el ángulo y la corriente, y nada más. No hay controlador a
bordo ni filtros: derivar la velocidad, elegir el signo y ajustar un modelo pasa
de este lado, donde se ve.

Se arma en cuatro pasos, y **cada paso ya sirve para algo**:

| | Configuración | Qué se agrega | Qué habilita |
|---|---|---|---|
| **A** | sólo el sensor | AS5600 en el bus I2C | el ángulo, el desenrollado, el ruido, el ritmo del muestreo |
| **B** | **con** puente en H, sección 2.1 | L298N y su fuente | mover el motor en los dos sentidos: lo tienen algunos bancos |
| **B′** | **con** actuador de un solo cuadrante, sección 2.2 | un transistor y un diodo | mover el motor para un lado: lo tienen los otros bancos |
| **C** | **sin** sensor de corriente | -- | todo lo anterior; `i` queda como un canal que no mide |
| **D** | **con** ACS712 | el sensor de corriente en A0 | ver el consumo&nbsp;⁽¹⁾ |

**Algunos bancos tienen el actuador B y otros el B′.** La celda de la sección 2.2
dice cuál tiene el que está sobre la mesa. Ninguno es un banco de segunda: todo lo
que pide el TP2 se mide con cualquiera de los dos, salvo la inversión del sentido
de giro, que sólo se puede hacer con B. El sketch y este notebook son los mismos
para los dos.

⁽¹⁾ Con reservas: la medición de corriente de este banco **no resuelve este motor**.
La nota al pie de la sección 4 dice exactamente por qué.

Después del recorrido, la sección 5 es **la API**: cada función que hace falta para
el TP2 --aplicar un comando, capturar, un escalón, una secuencia de escalones, la
velocidad, las unidades y el archivo--, con un ejemplo que se puede correr. La
identificación en sí --el modelo, el ajuste, la validación-- es el trabajo del TP,
y no está acá.

> ⚠️ **Con el motor conectado, el motor se mueve.** Revisar que el eje esté libre.

> Hay **un solo `dev`**, el de la celda que sigue, y el notebook se recorre de
> arriba abajo. Si se toca el cableado en el medio, volver a correr esa celda.

> **Para orientarse.** Poner `dev` en una celda muestra todo lo que la placa
> tiene: cada parámetro con su valor de ahora, su unidad, si se puede mover y una
> línea de qué es. `dev.describe('ang')` filtra por subsistema, y `dev.<TAB>`
> completa los nombres. No hay ninguna lista escrita de este lado: la placa
> declara la suya al conectarse.

In [ ]:
import sys, os
sys.path.insert(0, '../python')

import entorno
entorno.verificar()     # que el kernel sea el entorno dyc, con todo instalado

import numpy as np
import matplotlib.pyplot as plt

import ensayo
from banco_simulado import conseguir_banco

# --- cómo se ven los gráficos de este notebook -------------------------------
AZUL, NARANJA, AQUA, AMARILLO = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
TINTA, TENUE, NUBE = '#0b0b0b', '#52514e', '#c9c8c3'

plt.rcParams.update({
    'figure.figsize': (9, 3.6), 'figure.dpi': 110,
    'axes.grid': True, 'axes.axisbelow': True, 'grid.color': '#e6e5e1',
    'grid.linewidth': 0.8, 'axes.edgecolor': '#c9c8c3', 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelcolor': TENUE, 'axes.titlecolor': TINTA, 'axes.titlelocation': 'left',
    'axes.titleweight': 'medium', 'axes.titlepad': 10,
    'xtick.color': TENUE, 'ytick.color': TENUE, 'text.color': TINTA,
    'lines.linewidth': 1.8, 'legend.frameon': False, 'font.size': 10,
})

# El banco de verdad si el cable está enchufado; si no, uno simulado que lo dice.
dev = conseguir_banco(forzar_simulado=os.environ.get('HW_SIMULADO') == '1')
SIMULADO = getattr(dev, 'simulado', False)

# La calibración del sensor de este banco, si está medida. No es tema de este
# notebook --se mide en calibracion.ipynb-- pero sin ella el ángulo llega torcido,
# y una identificación de velocidad hereda la ondulación como si fuera del motor.
if not SIMULADO:
    try:
        import calib
        from bench import CALIBRACION
        if CALIBRACION.exists():
            calib.asegurar(dev, CALIBRACION)
            print('calibracion del sensor aplicada')
        else:
            print('sin calibracion del sensor: el angulo va crudo '
                  '(ver notebooks/calibracion.ipynb)')
    except Exception as exc:
        print(f'no se pudo aplicar la calibracion: {exc}')

## 0. El mapa

En el medio hay un UNO haciendo tres cosas a la vez, y conviene tenerlas separadas
en la cabeza porque cada una tiene su propio reloj:

1. **Muestrea el sensor a 5 kHz**, con el Timer2, período rígido. Una lectura del
   AS5600 por interrupción, sin esperar activamente.
2. **Pone el comando sobre el actuador y arma una fila a 500 Hz**: cada `loop_div`
   muestras --10 por omisión-- lee el ángulo y la corriente.
3. **Emite telemetría a 1 Mbaud**: una fila por período, 25 bytes, el 13 % del
   enlace. Es lo que llega a este notebook como un `DataFrame`.

```
   ┌────────────────────┐
   │    Arduino UNO     │   A4/A5 ──── I2C ─────►  AS5600      (configuración A)
   │                    │
   │  Timer2 → 5 kHz    │   9/6/7 ──── PWM+dir ─►  actuador ─► motor   (B, B′)
   │  filas  → 500 Hz   │
   │  USART  → 1 Mbaud  │   A0    ◄─── analógica   ACS712     (C y D)
   └────────┬───────────┘
            │ USB
            ▼
        Jupyter
```

Los pines, completos:

| Señal | Pin | Hace falta para | Si falta |
|---|---|---|---|
| AS5600 `SDA` | A4 | el ángulo | el ángulo queda congelado, las filas siguen saliendo |
| AS5600 `SCL` | A5 | ídem | ídem |
| AS5600 `VDD` / `GND` | 5V / GND | ídem | ídem |
| `ENA`, o la puerta del transistor | 9 (PWM) | mover el motor | se puede medir todo lo que no requiera movimiento |
| L298N `IN1` | 6 | el sentido, con un puente | con un transistor no va a ningún lado |
| L298N `IN2` | 7 | ídem | ídem |
| Salida del ACS712 | A0 | la corriente | `i` informa el ruido de una entrada al aire, y `bringup()` lo detecta |

**Periféricos de los que el sketch se apropia.** Vale saberlo antes de agregarle
algo al montaje, porque las funciones de Arduino que uno esperaría no están
disponibles:

| Recurso | Para qué | Qué deja de funcionar |
|---|---|---|
| Timer2 | el muestreador de 5 kHz | `analogWrite()` en 3 y 11, `tone()` |
| Timer1 | el PWM del actuador, con su propio TOP | `analogWrite()` en 9 y 10, `Servo` |
| ADC | se maneja a mano, canal 0, contra Vcc | `analogRead()` |
| USART | 1 Mbaud, el protocolo | `Serial` para cualquier otra cosa |
| TWI | `nI2C`, por interrupciones | `Wire` |
| Timer0 | -- | nada: `millis()` y el PWM de 5 y 6 andan como siempre (acá el 6 lo usa `IN1`, como salida digital) |

## 1. Configuración A: sólo el sensor

Sin puente, sin motor y sin medición de corriente. Es la configuración con la que
conviene empezar, y no es un juguete: acá ya se ve el sensor, el desenrollado, el
ruido y el ritmo del muestreo. Lo único que falta es que algo gire solo.

```
   ┌───────────────┐                            ┌─────────────┐
   │  Arduino UNO  │  A4 ──── SDA ──────────────│   AS5600    │
   │               │  A5 ──── SCL ──────────────│  (plaqueta) │
   │  Timer2:5 kHz │  5V ──── VDD ──────────────│             │
   │  USB: 1 Mbaud │ GND ──── GND ──────────────│             │
   └───────────────┘                            └──────┬──────┘
                                                       │
                                                imán diametral
                                             1-2 mm sobre el chip
```

**El imán es parte del sensor, no un accesorio.** Tiene que estar magnetizado
**diametralmente** --los polos enfrentados a lo ancho, no norte arriba y sur
abajo--, girar sobre la cara del chip a un par de milímetros, y estar centrado con
el eje de giro. La hoja de datos pide un cuarto de milímetro de excentricidad; lo
que sobra de eso no aparece como ruido sino como una función fija del ángulo que se
repite vuelta tras vuelta. Eso es el tema completo de `calibracion.ipynb`.

**Las plaquetas comerciales de AS5600 traen su propio regulador y los pull-ups del
bus**, así que se enchufan a 5 V y andan. Un chip pelado en modo 3,3 V necesita
adaptación de niveles y sus propias resistencias de pull-up.

**El AS5600 tiene un filtro adentro**, y viene mal puesto para esto: arranca en
16x, que son 2,2 ms de retardo. El sketch lo pasa a 2x, 0,286 ms, una vez al
arrancar. Ese retardo se identificaría después como si fuera un tiempo muerto del
motor, así que no es una perilla: se lo deja al mínimo y listo.

**El muestreo no se detiene si el sensor no está.** Si el AS5600 no contesta, el
muestreador pasa a sondear el bus dos veces por segundo en lugar de cinco mil, y
todo lo demás --el período, la telemetría, los parámetros-- sigue igual. Sirve para
probar la cadena completa (compilar, grabar, capturar, graficar) antes de tener el
sensor sobre la mesa.

`bringup()` es la verificación del equipo, subsistema por subsistema. Con
`motor=False` saltea todo lo que haría girar el eje, que es exactamente lo que
corresponde en esta configuración.

In [ ]:
dev.bringup(motor=False)

Lo que hay que mirar en esa salida, en orden:

- **`muestreo`**: la frecuencia real contra la nominal, medida con el reloj de esta
  computadora y no con el contador de la placa --que avanza una vez por período
  *atendido* y por eso daría siempre por bueno lo que hay que detectar.
- **`margen de tiempo`**: cuánto del período se consume en el peor caso. Sin margen
  alguno la placa está por empezar a perder filas.
- **`iman`**: el AGC del propio AS5600. Contra 0 o contra 255 el imán está a la
  distancia equivocada, y ahí no hay calibración que arregle nada.
- **`bus i2c`**: errores de transferencia y desbordes. Un puñado de desbordes por
  segundo es normal --el diagnóstico del imán lee un registro extra dos veces por
  segundo y esa lectura no entra en 200 us--; lo que no es normal es que el bus no
  llegue de manera sostenida.

Y ahora el sensor en vivo. Hay que **girar el imán con la mano** mientras la celda
corre. `y_uw` es el ángulo *desenrollado*: sigue contando a través de la vuelta de
4096 cuentas en lugar de saltar a cero, así que un eje que gira sin parar da una
recta que no para. `y_raw` es la cuenta cruda, adentro de la vuelta: la que salta.

In [ ]:
df = dev.capture(3.0)

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'], df['y_uw'] - df['y_uw'].iloc[0], color=AZUL)
a.set_ylabel('y_uw [grados]')
b.plot(df['t'], df['y_raw'], lw=0.8, color=TENUE)
b.set_ylabel('y_raw [grados]'); b.set_xlabel('t [s]')
a.set_title('Girar el imán con la mano')
plt.show()

print(f'se movió {df["y_uw"].max() - df["y_uw"].min():.1f} grados')
print(f'ruido entre muestras consecutivas: {df["y_uw"].diff().std():.3f} grados '
      f'({df["y_uw"].diff().std() / 0.0879:.2f} cuentas)')

## 2. Configuración B: con actuador

Ahora el motor. El UNO no maneja un motor: maneja una llave, y la llave maneja al
motor con su propia fuente. Hay dos actuadores posibles, según el banco: un puente en H, que acciona
en los dos sentidos (2.1), y uno de un solo cuadrante (2.2). Conviene leer los dos: la
celda del final de 2.2 dice cuál tiene cada banco.

### 2.1 Un puente en H: los dos sentidos

```
   ┌───────────────┐                     ┌──────────────────┐
   │  Arduino UNO  │  9 ──── ENA ────────│      L298N       │      ┌─────────┐
   │               │  6 ──── IN1 ────────│   puente en H    │ OUT1─│  motor  │
   │  Timer1: PWM  │  7 ──── IN2 ────────│                  │ OUT2─│ + imán  │
   │               │ GND ─────┬──────────│ GND       +Vmot  │      └─────────┘
   └───────────────┘          │          └───────┬─────┬────┘
                              └── masa común ────┘     │
                                                       └── fuente del motor
                                                           (NO el USB)
```

**Tres reglas de cableado, y las tres se pagan caras si se saltean.** La fuente del
motor es propia y no el USB: el UNO da la lógica, nunca la potencia. Las masas van
unidas, porque si no las señales de control no tienen contra qué medirse. Y el
actuador es el único que toca los bornes del motor.

**El reparto de los tres pines no es arbitrario.** `ENA` lleva la *magnitud* por
PWM y el par `IN1`/`IN2` el *sentido*. Lo que eso compra es que el sketch pueda
**apagar el puente antes de cambiar de sentido**: baja `ENA` --que abre las cuatro
llaves con una sola escritura--, recién entonces mueve `IN1` e `IN2`. Con el
comando en cero `ENA` queda en bajo, el puente abierto y el motor en punto muerto:
**no frena el eje, sólo deja de empujarlo**.

**El PWM va a 1 kHz, y es una concesión al L298N.** En abstracto conviene modular
más rápido, pero es un puente de Darlington bipolares: cae unos 2 V y tarda unos
2 us en conmutar, y a 20 kHz lo que se pierde en cada transición se lleva una
fracción grande de un tiempo de encendido que ya venía escaso. **Medido: a 20 kHz
el motor no arranca y a 1 kHz anda.** Es una constante del sketch (`PWM_TOP`).

**El signo no es un parámetro.** Si un comando positivo hace *bajar* el ángulo, es
de qué lado están los cables del motor y de qué lado mira el imán; en lazo abierto
eso se resuelve al procesar, con `ensayo.signo()`, y no en la placa.

**Con un L298N**, hay además una manera de accionarlo que deja la
planta lineal con un solo comando: frenar en lugar de soltar, con `ENA` en alto y
el PWM sobre la entrada del sentido --la otra en cero--, de modo que en la parte
baja del ciclo el puente cortocircuita el motor en vez de abrirlo. Lo que no
conviene es modular bipolar a 1 kHz.

### 2.2 Un solo cuadrante: el transistor

Un transistor y un diodo, lo mínimo que mueve un motor:

```
                       +Vmot
                         │
                  ┌──────┴──────┐
               [motor]     [diodo de rueda libre]   el cátodo (la banda) va
                  │             │                   del lado de +Vmot
                  └──────┬──────┘
                         │
   UNO 9 ──[1k]──► G ┌───┴────┐
                     │ llave  │  del lado de masa
                     └───┬────┘
                         │
                        GND ──── común con el UNO
```

**El diodo no es opcional**: al cortar la corriente, la inductancia del motor la
sigue empujando, y sin un camino de retorno esa energía aparece como un pico de
tensión sobre el transistor.

**Qué cambia respecto del puente**, y es lo que cualquier modelo de un banco B′ tiene
que contemplar:

- **Empuja y no frena.** Cuando el transistor se abre la corriente se descarga por
  el diodo, y cuando se extingue no hay nada que la invierta. Bajar la velocidad lo
  hace el rozamiento solo, así que **bajar tarda bastante más que subir**.
- **El sentido está fijado en cobre.** `IN1` e `IN2` no van a ningún lado: un
  comando negativo sale por el pin 9 con el mismo módulo y **empuja para el mismo
  lado**. La celda de abajo lo muestra.
- **Con el comando en cero el eje sigue girando** muchos segundos. Todo ensayo tiene
  que esperar a que pare: `ensayo.esperar_quieto(dev)`.
- **La corriente se corta antes de terminar el período** a velocidades medias --la
  inductancia del motor es chica contra un período de 1 ms--, y eso hace que la
  velocidad no sea proporcional al comando: la curva estática del TP se dobla.

**Qué sigue funcionando completo**, que es casi todo: **todo lo que pide el TP2**
salvo la inversión del sentido de giro, y **la calibración del sensor** (`calibracion.ipynb`), que mide soltando el motor y
dejándolo desacelerar por rozamiento, que es justo lo que sabe hacer un cuadrante.

**¿Cuál tiene este banco?** La celda que sigue aplica un comando positivo y uno
negativo y mira para dónde gira el eje. Con un puente en H (**B**) las dos
velocidades salen con signos opuestos; con un transistor (**B′**), con el mismo.

In [ ]:
print('lo que hace el eje con un comando positivo y con uno negativo:')
velocidades = []
for u in (+150, -150):
    ensayo.esperar_quieto(dev)
    dev.ctl_uff = u
    dev.capture(2.0, warn=False)                       # que llegue al régimen
    df = dev.capture(0.5, warn=False)
    dev.rest()
    _, w = ensayo.velocidad(df, ventana=0)
    velocidades.append(np.mean(w))
    print(f'  ctl_uff = {u:+4d}   →   u = {df["u"].mean():+6.1f} en la placa, '
          f'{np.mean(w):+7.1f} rad/s')

if velocidades[0] * velocidades[1] < 0:
    print('\nsignos opuestos: este banco es B, un puente en H. El comando elige el sentido.')
else:
    print('\nmismo signo: este banco es B′, un solo cuadrante. El sentido está en los cables,')
    print('y un comando negativo empuja igual que uno positivo.')

### 2.3 La verificación completa

`bringup()`, ahora con el motor. Espera a que el eje esté quieto, lo acciona unas
décimas de segundo, y se fija que haya vuelto movimiento o corriente. Si un comando
positivo hace bajar el ángulo lo dice como nota --no es una falla: se resuelve al
procesar--, y si el pico de corriente del arranque no se despega del ruido del
canal, también lo dice.

In [ ]:
dev.bringup()                    # OJO: esto mueve el motor

## 3. Configuración C: sin medición de corriente

Es la configuración de la mayoría de los bancos, y no se pierde casi nada. Sin nada
conectado a A0:

- el canal `i` sigue apareciendo en cada fila, informando el ruido de una entrada
  al aire;
- `bringup()` **lo detecta y lo dice**: la entrada queda contra un riel del ADC, y
  eso es una *ausencia*, no un offset. La diferencia importa, porque calibrar un
  cero ahí dejaría un canal que informa ceros perfectos sin haber medido nada;
- lo que no se puede hacer es mirar el pico de arranque, ni usar la corriente como
  evidencia de que el motor está haciendo algo.

Todo el resto --el ángulo, la identificación de la planta mecánica-- funciona igual.

## 4. Configuración D: con un ACS712

El ACS712 es un sensor de corriente de efecto Hall: la corriente atraviesa una
pista interna, el campo que genera se mide del otro lado de un aislamiento, y la
salida es una tensión analógica que va a A0.

**Dónde se lo inserta cambia lo que mide.** En la alimentación del actuador mide el
consumo, que es siempre positivo; en serie con un borne del motor mide la corriente
del motor, que con un solo cuadrante también es siempre positiva.

**El ADC lee contra Vcc**, y no por gusto: un ACS712 es bipolar y ratiométrico,
reposa en la mitad de su alimentación, y medido contra la misma tensión que lo
alimenta queda en media escala por construcción. Contra la referencia interna de
1,1 V saturaría en reposo.

| | 1 cuenta de 12 bits | Con 185 mV/A |
|---|---|---|
| clon con LGT8F328P (ADC de 12 bits) | 1,22 mV | 6,6 mA |
| UNO (ADC de 10 bits, corrido dos lugares) | 4,89 mV | 26,4 mA |

**Hay un cero que se puede medir y una ganancia que no.** El cero tiene una
condición conocida --con el actuador abierto no circula corriente-- así que
`dev.zero_current()` lo mide y lo resta, y `bringup()` ya lo corre solo. La
ganancia es otra cosa: haría falta una **corriente conocida**. Un tester en serie
con el motor, una vez; el número vive en `SENSE_MV_PER_A`, en el sketch.

La celda que sigue mira la corriente durante un escalón y la compara con su propio
ruido en reposo.

In [ ]:
ensayo.esperar_quieto(dev)
dev.zero_current()

reposo = dev.capture(1.0, warn=False)
ruido = reposo['i'].std()

df = dev.step('ctl_uff', 200, pre=0.3, post=1.5, back=0)
dev.rest()

# Un promedio de 100 ms es lo mejor que se puede pedirle a este canal: una
# muestra suelta es casi toda ruido.
n = int(0.1 / dev.dt)
suave = np.convolve(df['i'], np.ones(n) / n, mode='same')

plt.plot(df['t'], df['i'], color=NUBE, lw=0.8, label='cada muestra')
plt.plot(df['t'], suave, color=NARANJA, label='promedio de 100 ms')
plt.axvline(0, color=TENUE, lw=0.9, ls='--')
plt.axhline(0, color=TENUE, lw=0.9)
plt.xlabel('t [s]'); plt.ylabel('i [mA]'); plt.legend()
plt.title('Corriente durante el escalón')
plt.show()

regimen = df[df['t'] > 1.0]['i'].mean()
print(f'ruido en reposo    {ruido:.0f} mA RMS por muestra')
print(f'régimen            {regimen:+.0f} mA de promedio')
print(f'pico               {df["i"].max():.0f} mA, que contra el ruido son '
      f'{df["i"].max() / ruido:.1f} desvíos')
if df['i'].max() < 5 * ruido:
    print('\nel pico no se despega de cinco veces el ruido: este canal NO resuelve la '
          'corriente de este motor. Sirve para ver que hay algo, no para medirlo.')

### ⁽¹⁾ Nota al pie: por qué esta medición no resuelve este motor

Es la parte del banco que hay que mirar con desconfianza, y vale la pena que quede
escrito por qué. Ninguna es un cable flojo:

1. **El motor consume decenas de mA y el sensor es de amperes.** Un ACS712 de 5 A
   da 185 mV/A: 50 mA son 9 mV, que en el UNO no llegan a dos cuentas del ADC.
2. **El ruido del sensor es más grande que la señal.** Medido en este banco: 91 mA
   RMS por muestra, del orden de toda la corriente de régimen. Promediando se baja,
   pero promediar es filtrar, y un filtro largo borra justo el transitorio que se
   quería ver.
3. **Se muestrea la forma de onda en una fase fija.** El ADC convierte una vez por
   tick de 200 us y el PWM va a 1 kHz, del mismo cristal: la fase **no deriva
   nunca**. Lo que se mide no es el promedio de la corriente sino siempre el mismo
   instante del período, y con un solo cuadrante la corriente se extingue antes de
   que termine: el sesgo depende del ciclo de trabajo y no se promedia con el tiempo.
4. **La ganancia no es verificable desde acá.** Con el actuador abierto la corriente
   es cero y eso alcanza para el offset, pero no hay ninguna condición conocida de
   corriente *distinta* de cero. Hasta que alguien ponga un tester en serie, la
   escala de este canal es una hipótesis.

**Qué se puede hacer igual con este canal**: ver *que* hay corriente, y comparar
órdenes de magnitud. **Qué no**: usarlo para nada cuantitativo --un $K_t$, un par de
rozamiento, un modelo eléctrico--. Y así como está **es un buen ejercicio**: medir
con un tester, comparar, y decidir qué sensor haría falta.

## 5. La API para el TP2

Todo se maneja con tres ideas: **los parámetros son atributos**, **las capturas
devuelven un `DataFrame`** y **las unidades son unidades reales**. Esta sección
recorre, en el orden en que aparecen en un ensayo, todas las funciones que hacen
falta para el TP2:

| Para | Se usa | Sección |
|---|---|---|
| conectarse y verificar el equipo | `conseguir_banco()`, `dev.bringup()`, `dev` | 5.1 |
| aplicar un comando de PWM | `dev.ctl_uff`, `dev.rest()` | 5.2 |
| empezar con el eje parado | `ensayo.esperar_quieto(dev)` | 5.2 |
| el cero de la corriente | `dev.zero_current()` | 5.2 |
| grabar señales durante un tiempo | `dev.capture(segundos)`, `df.attrs`, `dev.dt` | 5.3 |
| una respuesta al escalón $u_0 \to u_1$ | `dev.step(...)` | 5.4 |
| una serie de escalones, o una escalera | `dev.capture(..., events=[...])` | 5.5 |
| la velocidad a partir del ángulo | `ensayo.velocidad(df, ventana)`, `ensayo.signo(df)` | 5.6 |
| $t$, $u$, $\theta$, $\omega$, $i$ en unidades de un modelo | `ensayo.normalizar(df)` | 5.7 |
| guardar y leer los datos | `ensayo.guardar(df, ruta)`, `ensayo.cargar(ruta)` | 5.7 |

Cualquier función se puede consultar desde una celda con `help(dev.step)` o
`help(ensayo.velocidad)`.

### 5.1 Conseguir el banco

La primera celda de este notebook ya lo hizo:

```python
dev = conseguir_banco()   # compila si cambió el sketch, graba si cambió el binario,
                          # reabre el enlace (lo que resetea la placa), y si no hay
                          # placa cae a un banco simulado, avisando fuerte
```

Si se toca el cableado, o se desenchufa la placa, volver a correr la primera celda.
`dev.bringup()` (sección 2.3) es la verificación completa, y lo que conviene correr
ante cualquier duda.

**Los parámetros son atributos, y la placa los enumera sola.** Se leen y se
escriben como cualquier atributo, y cada uno viaja a la placa en el momento. Poner
`dev` en una celda muestra la tabla que realmente hay: cada parámetro, su valor de
ahora, su unidad y qué es. `dev.describe('ang')` filtra por subsistema.

| Prefijo | De qué es | Qué significa |
|---|---|---|
| `ctl_` | el comando | una perilla: se fija, es una decisión del experimento |
| `ang_` | el sensor de ángulo | una lectura: la placa la publica |
| `cur_` | la medición de corriente | el cero y los contadores |
| `loop_` | el reloj del muestreo | `loop_div` fija la frecuencia de las filas |

In [ ]:
dev

### 5.2 Aplicar un comando, y empezar desde el reposo

El comando es **`dev.ctl_uff`**, en **cuentas de PWM**: de 0 a 255 para un sentido,
y con un puente en H (B) de -255 a 0 para el otro. El TP expresa
$u$ en por ciento, así que se convierte a mano: $u[\%] = 100 \cdot \text{ctl\_uff} / 255$.

```python
dev.ctl_uff = round(40 * 255 / 100)   # 40 %: 102 cuentas
dev.rest()                            # comando en cero: donde termina todo experimento
```

**Qué actuador tiene el banco cambia qué comandos significan algo** (sección 2.2):

| | Banco B, puente en H | Banco B′, transistor |
|---|---|---|
| comando positivo | gira para un lado | gira |
| comando negativo | gira para el otro lado | **gira para el mismo lado** que uno positivo |
| comando en cero | el puente se abre: el eje no frena, sigue por inercia | ídem |
| invertir el giro (TP, 8.4 y 10) | se puede: un escalón de `+u` a `-u` | no se puede |

Todo lo demás del TP se hace con comandos de 0 a 255, y es igual en los dos.

**`ensayo.esperar_quieto(dev)`** pone el comando en cero y espera, midiendo, a que el
eje pare. **Va antes de cada ensayo**: con el comando en cero el motor no frena y
sigue girando muchos segundos, y un ensayo que arranca antes mide la cola del
anterior. Devuelve los segundos que esperó.

**`dev.zero_current()`** toma la corriente de ahora como el cero, con el actuador
abierto. Conviene correrlo antes de un ensayo en el que se vaya a mirar `i`.

In [ ]:
espera = ensayo.esperar_quieto(dev)
dev.zero_current()
print(f'eje parado despues de {espera:.1f} s; cero de corriente en {dev.cur_zero} cuentas')

### 5.3 Capturar

**`dev.capture(segundos)`** graba y devuelve un `DataFrame` con una fila por período
de muestreo. Las columnas son los canales de la placa:

| Columna | Qué es | Unidad |
|---|---|---|
| `t` | el reloj de la placa; en un escalón de `dev.step()`, `t = 0` es el escalón | s |
| `y_uw` | el ángulo **desenrollado**: sigue contando a través de la vuelta | grados |
| `y_raw` | el ángulo crudo del sensor, adentro de la vuelta | grados |
| `u` | el comando que salió al actuador | cuentas, -255 a 255 |
| `i` | la corriente | mA |

**`t` no empieza en cero** en una captura común: cuenta desde que arrancó la placa.
Para medir desde el comienzo de la captura, `df['t'] -= df['t'].iloc[0]`.

**El período de muestreo** $T_s$ es `dev.dt`, en segundos: 2 ms, o sea 500 Hz. Se
cambia con `dev.loop_div`, que divide el muestreador de 5 kHz: `dev.loop_div = 5` da
1 kHz. Más rápido no siempre es mejor: con el sensor cuantizado, la velocidad
calculada entre dos muestras más cercanas es más ruidosa (ver 5.6).

**`df.attrs` trae la salud de esa captura en particular.** Toda captura además avisa
si perdió períodos o descartó filas, porque una serie temporal a la que le faltan
muestras se ve exactamente igual que una sana. `warn=False` apaga el aviso.

In [ ]:
df = dev.capture(1.0, warn=False)

print('columnas:', ', '.join(df.columns))
print(f'{len(df)} filas; Ts = dev.dt = {dev.dt * 1e3:.1f} ms, o sea {1 / dev.dt:.0f} Hz')
print()
print('salud de esta captura:')
print(f'  períodos perdidos      {df.attrs["missed"]}')
print(f'  filas descartadas      {df.attrs["drops"]}')
print(f'  desbordes del sensor   {df.attrs.get("sovr")}')
df.head()

### 5.4 Un escalón: `dev.step()`

```python
df = dev.step('ctl_uff', u1, pre=0.5, post=4.0, back=0)
```

Captura `pre` segundos con el comando que haya, lo cambia a `u1`, captura `post`
segundos más y, si se da `back`, lo deja en ese valor al terminar. La placa informa
el período exacto en el que cayó el cambio, así que **`t = 0` es el escalón mismo**,
con precisión de una muestra.

Para el escalón $u_0 \to u_1$ del TP (sección 3.2) hay que llegar primero al
régimen de $u_0$: fijar `ctl_uff = u0` y descartar una captura de unos segundos.
Para medir en varios puntos de operación (sección 8.1 del TP) se repite esto en un
`for`, con un `esperar_quieto` antes de cada uno.

In [ ]:
U0, U1 = round(40 * 255 / 100), round(60 * 255 / 100)    # 40 % → 60 %

ensayo.esperar_quieto(dev)
dev.zero_current()
dev.ctl_uff = U0
dev.capture(4.0, warn=False)                              # llegar al régimen de u0; se descarta
df = dev.step('ctl_uff', U1, pre=0.5, post=4.0, back=0)

datos = ensayo.normalizar(df)                             # ver 5.7

fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6.4))
a.plot(datos['t'], datos['u'], color=TENUE, label='u');        a.set_ylabel('u [%]')
b.plot(datos['t'], datos['omega'], color=AZUL, label='ω');     b.set_ylabel('ω [rad/s]')
c.plot(datos['t'], datos['i'] * 1000, color=NARANJA, lw=0.8, label='i'); c.set_ylabel('i [mA]')
c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color=TENUE, lw=0.9, ls='--')
    ax.legend(loc='upper left')
a.set_title('Escalón de u = 40 % → 60 %')
plt.show()

### 5.5 Una secuencia de comandos: `events`

`dev.step()` es un caso particular de algo más general: **`dev.capture(segundos,
events=[...])`**, donde cada evento es una tupla `(segundos, 'ctl_uff', valor)` que
fija el comando a esos segundos del comienzo de la captura. Igual que en el
escalón, la placa registra el período exacto de cada cambio.

Con eso se arma cualquier señal de entrada escalonada, por ejemplo:

- **una serie de escalones de subida y bajada**, para validar un modelo contra una
  señal que no se usó para ajustarlo (sección 5 del TP);
- **una escalera lenta desde cero**, para la zona muerta y la curva estática
  $\omega = f(u)$ (sección 8.2 del TP).

Y, **sólo en un banco B**, la inversión del sentido de giro: un evento con un valor
negativo, por ejemplo `[(0.0, 'ctl_uff', 150), (3.0, 'ctl_uff', -150)]`. En un banco
B′ eso no invierte nada: el motor sigue girando para el mismo lado.

El `u` que vuelve en la captura es el que realmente salió al actuador, así que es
el que hay que usar en la simulación.

In [ ]:
# Una serie de escalones: 2 s en cada nivel.
niveles = [40, 70, 50, 80, 30]                            # en %
eventos = [(2.0 * k, 'ctl_uff', round(p * 255 / 100)) for k, p in enumerate(niveles)]

ensayo.esperar_quieto(dev)
serie = dev.capture(2.0 * len(niveles), events=eventos)
serie['t'] -= serie['t'].iloc[0]                         # que el tiempo empiece en cero
dev.rest()

d = ensayo.normalizar(serie)
fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(d['t'], d['u'], color=TENUE);  a.set_ylabel('u [%]')
b.plot(d['t'], d['omega'], color=AZUL); b.set_ylabel('ω [rad/s]'); b.set_xlabel('t [s]')
a.set_title('Una serie de escalones con events')
plt.show()

In [ ]:
# Una escalera lenta desde cero, para ver dónde arranca el motor: de a 4 %, 1,5 s cada escalón.
porcentajes = np.arange(0, 41, 4)
eventos = [(1.5 * k, 'ctl_uff', round(p * 255 / 100)) for k, p in enumerate(porcentajes)]

ensayo.esperar_quieto(dev)
escalera = dev.capture(1.5 * len(porcentajes), events=eventos)
escalera['t'] -= escalera['t'].iloc[0]
dev.rest()

d = ensayo.normalizar(escalera, ventana=0.2)              # una ventana larga: acá interesa el régimen
fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(d['t'], d['u'], color=TENUE);  a.set_ylabel('u [%]')
b.plot(d['t'], d['omega'], color=AZUL); b.set_ylabel('ω [rad/s]'); b.set_xlabel('t [s]')
a.set_title('Una escalera lenta desde cero')
plt.show()

### 5.6 De ángulo a velocidad: `ensayo.velocidad()`

La placa **no mide la velocidad**: la calcula este lado, a partir del ángulo, y a
propósito. Así se ve qué se hizo, se puede cambiar, y se puede discutir, que es lo
que pide la sección 2.1 del TP.

```python
t, w = ensayo.velocidad(df, ventana=0.02)   # rad/s
```

Hace dos cosas, en este orden:

1. **La diferencia hacia atrás**, $\omega[k] = (\theta[k] - \theta[k-1]) / T_s$: lo que
   uno escribiría en un microcontrolador.
2. **Un promedio móvil centrado** de `ventana` segundos. Centrado quiere decir que no
   atrasa la señal --un filtro causal la atrasaría, y ese atraso se confundiría con
   un tiempo muerto del motor--, pero sí redondea las esquinas de un escalón: la
   ventana tiene que ser corta contra la constante de tiempo que se quiere ver. Y
   como mira hacia adelante, sirve para procesar una captura, no para un lazo de
   control.

Con `ventana=0` no se promedia, y se ve **la cuantización desnuda**: una cuenta del
sensor (0,088°) en un período de 2 ms son 0,77 rad/s. Devuelve una muestra menos que
la captura.

**El signo.** Si un comando positivo hace *bajar* el ángulo, la velocidad sale
negativa. Es cosa del cableado y de qué lado mira el imán, no del motor:
`ensayo.signo(df)` lo detecta (+1 o -1), y `ensayo.normalizar()` lo aplica solo.

In [ ]:
# El mismo escalón de 5.4, con tres ventanas distintas.
s = ensayo.signo(df)

plt.figure(figsize=(9, 4))
for ventana, color, ancho in ((0, NUBE, 0.8), (0.02, AZUL, 1.4), (0.2, NARANJA, 1.8)):
    t, w = ensayo.velocidad(df, ventana=ventana, signo_banco=s)
    plt.plot(t, w, color=color, lw=ancho, label=f'ventana = {ventana * 1e3:.0f} ms')
plt.axvline(0, color=TENUE, lw=0.9, ls='--')
plt.xlim(-0.2, 1.5)
plt.xlabel('t [s]'); plt.ylabel('ω [rad/s]'); plt.legend()
plt.title('La misma captura, derivada con tres ventanas')
plt.show()

print(f'signo del banco: {s:+d}; Ts = {dev.dt * 1e3:.0f} ms; '
      f'una cuenta del sensor por período = {np.deg2rad(360 / 4096) / dev.dt:.2f} rad/s')

### 5.7 Unidades de un modelo, y el archivo

**`ensayo.normalizar(df)`** devuelve la captura con las columnas que pide el TP, en
las unidades en las que se escribe un modelo:

| Columna | Qué es | Unidad |
|---|---|---|
| `t` | tiempo; en un escalón, `t = 0` es el escalón | s |
| `u` | el comando | % de PWM |
| `theta` | la posición, desenrollada y con el signo corregido | rad |
| `omega` | la velocidad, de `ensayo.velocidad()` | rad/s |
| `i` | la corriente | A |

Acepta los mismos `ventana` y `signo_banco` que `velocidad()`.

**`ensayo.guardar(df, ruta)`** escribe eso mismo en un CSV, y **`ensayo.cargar(ruta)`**
lo lee de vuelta. Es el formato que se puede importar en PID Tuner o APMonitor, y
conviene guardar **cada** ensayo apenas se mide: medir de nuevo nunca da exactamente
lo mismo.

In [ ]:
ruta = ensayo.guardar(df, 'datos/escalon_40_60.csv')     # ruta relativa a notebooks/
leido = ensayo.cargar(ruta)

print(f'guardado en {ruta.resolve()}')
leido.head()

### Dos advertencias

Son las que arruinan un ensayo sin dejar rastro en el gráfico:

**Un ensayo que no espera al eje mide al anterior.** Con el comando en cero el motor
sigue girando muchos segundos. `ensayo.esperar_quieto(dev)` antes de cada uno.

**El sensor sin calibrar mete una ondulación de velocidad que parece del motor.**
Enganchada al ángulo y repetida vuelta tras vuelta, al derivar sale como una
oscilación perfectamente creíble. `calibracion.ipynb` la mide y la corrige, y la
primera celda de este notebook la aplica si está medida.